In [1]:
import os
import warnings
import tqdm
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=pd.errors.PerformanceWarning)

In [2]:
%load_ext autoreload
%autoreload 2

## Select data

In [ ]:
datafolder = "../data"
spadl_h5 = os.path.join(datafolder, "women-statsbomb3.h5")

In [4]:
games = pd.read_hdf(spadl_h5, "games")
print("nb of games:", len(games))

nb of games: 126


## Compute features

In [ ]:
# ── Pitch / event constants ───────────────────────────────────────────────────
# StatsBomb coordinate system: possession team always attacks toward x = 120
ATT_THIRD_X = 80.0          # boundary of the attacking third

TOUCH_TYPES = {
    "Pass", "Carry", "Shot", "Dribble", "Ball Recovery",
    "Clearance", "Interception", "Miscontrol", "Dispossessed",
    "Goal Keeper", "Foul Committed", "Error", "Block",
}
DEFENSIVE_TYPES = {
    "Pressure", "Duel", "Block", "Interception", "Clearance", "Ball Recovery",
}
EXCLUDE_TYPES = {
    "Starting XI", "Half Start", "Half End", "Tactical Shift",
    "Substitution", "Bad Behaviour", "Referee Ball-Drop",
}

# ── Helper functions ───────────────

def parse_sub(extra, event_type):
    """Return the event-specific sub-dict from the extra field."""
    key = {"Pass": "pass", "Carry": "carry", "Shot": "shot"}.get(event_type, "")
    return extra.get(key, {}) if isinstance(extra, dict) and key else {}


def compute_ball_distances(att_events):
    """Return (D_t, D_f): total ball travel distance and forward (Δx>0) distance.

    - Passes: use pre-computed pass.length (StatsBomb units, Euclidean)
    - Carries: compute Euclidean distance from location to carry.end_location
    Forward distance counts only x-axis gains (possession team attacks → x=120).
    """
    D_t = D_f = 0.0
    for _, r in att_events.iterrows():
        loc = r["location"]
        sub = parse_sub(r["extra"], r["type_name"])
        if r["type_name"] == "Pass":
            D_t += sub.get("length", 0) or 0
            end = sub.get("end_location")
            if end and isinstance(loc, list):
                dx = end[0] - loc[0]
                if dx > 0:
                    D_f += dx
        elif r["type_name"] == "Carry":
            end = sub.get("end_location")
            if end and isinstance(loc, list):
                dx, dy = end[0] - loc[0], end[1] - loc[1]
                D_t += np.hypot(dx, dy)
                if dx > 0:
                    D_f += dx
    return D_t, D_f


def parse_freeze_frame(ff):
    """Split a 360 freeze frame into three player-location lists.

    Returns:
        att_field : [x, y] list – possession-team outfield players (keeper=False, teammate=True)
        def_field : [x, y] list – defending-team outfield players  (keeper=False, teammate=False)
        att_all   : [x, y] list – all possession-team players incl. GK (for centroid)
    """
    att_field, def_field, att_all = [], [], []
    if not isinstance(ff, list):
        return att_field, def_field, att_all
    for p in ff:
        loc = p.get("location")
        if not (isinstance(loc, list) and len(loc) >= 2):
            continue
        is_team   = p.get("teammate", False)
        is_keeper = p.get("keeper",   False)
        if is_team:
            att_all.append(loc[:2])
            if not is_keeper:
                att_field.append(loc[:2])
        elif not is_keeper:
            def_field.append(loc[:2])
    return att_field, def_field, att_all


def _cv(values):
    """Coefficient of variation (std / mean); NaN if fewer than 2 values."""
    if len(values) < 2:
        return np.nan
    a  = np.asarray(values, dtype=float)
    mu = a.mean()
    return float(a.std() / mu) if mu != 0 else np.nan


def _spread_accel(vals, times):
    """Mean absolute rate of change of team length/width ratio (|Δratio/Δt|)."""
    if len(vals) < 2:
        return np.nan
    sv = np.asarray(vals,  dtype=float)
    st = np.asarray(times, dtype=float)
    dt = np.diff(st)
    ok = dt > 0
    acc = np.abs(np.diff(sv)[ok] / dt[ok])
    acc = acc[acc < 10.0]
    return float(np.mean(acc)) if len(acc) > 0 else np.nan


def _inter_pass_interval(att_events, passes):
    '''
    PoP(ai) = Tai - Tai-1
    Tai: 当前球员传球时间（球离脚）
    Tai-1: 上一个队友传球时间（球离脚）
    '''
    df = att_events.copy().reset_index(drop=True)

    if 'timestamp' not in df.columns:
        return np.nan
        
    if not pd.api.types.is_timedelta64_dtype(df["timestamp"]):
        df["timestamp"] = pd.to_timedelta(df["timestamp"])

    intervals = []

    if 'timestamp' not in passes.columns:
        return np.nan
        
    if not pd.api.types.is_timedelta64_dtype(passes["timestamp"]):
        passes["timestamp"] = pd.to_timedelta(passes["timestamp"])
        
    for idx, current_pass in passes.iterrows():
        current_player_id = current_pass["player_id"]
        current_team_id = current_pass["team_id"]
        current_possession = current_pass["possession"]
        current_time = current_pass["timestamp"]  # Tai

        # find previous events
        previous_events = df[
            (df.index < idx) & 
            (df["possession"] == current_possession) 
        ]
        
        if len(previous_events) == 0:
            continue

        # situation 1: teammate's pass
        teammate_passes = previous_events[
            (previous_events["type_name"] == "Pass") &
            (previous_events["team_id"] == current_team_id) &
            (previous_events["player_id"] != current_player_id)
        ]

        # situation 2: current player's interception
        player_interceptions = previous_events[
            (previous_events["type_name"] == "Interception") &
            (previous_events["player_id"] == current_player_id)
        ]

        # merge two types, take the latest one
        valid_previous = pd.concat([teammate_passes, player_interceptions])
        
        if len(valid_previous) > 0:
            valid_previous = valid_previous.sort_index()
            previous_action = valid_previous.iloc[-1]  # Tai
            previous_time = previous_action["timestamp"]  # Tai-1

            # calculate PoP = Tai - Tai-1
            delta = (current_time - previous_time).total_seconds()

            # filter outliers
            if 0 <= delta <= 30:
                intervals.append(delta)
                
    return float(np.mean(intervals)) if intervals else np.nan

In [ ]:
def compute_pace_metrics(events):
    """Compute possession-level game pace metrics from raw StatsBomb events.

    Appendix metrics
    ────────────────
    Activity Intensity
      passes_per_second          : N_passes / T_p
      distance_per_second        : D_t / T_p   (ball travel distance)
      distance_per_touch         : D_t / T_touches
      inter_pass_interval        : mean(t_pass – t_receipt) per player (Bransen & Davis, 2025)

    Directional Intent
      attacking_transition_time  : t_{final-third} – t_{start}
      time_to_shot               : t_{shot} – t_{start}
      team_centroid_speed        : dC/dt  (from 360 freeze frames, dt > 100 ms)
      forward_progression_rate   : D_f / D_t
      attacking_line_variability : CV of last outfield attacker x-position over time
      team_spread_acceleration   : |d/dt(Δlength/width)| (from 360)
      inter_team_longitudinal_dist : mean(x_def) – mean(x_att) across 360 frames

    Constraint
      pressing_frequency         : N_defensive / T_p
      spatial_compression_x      : mean SD of attacker x-positions (from 360)
      spatial_compression_y      : mean SD of attacker y-positions (from 360)
      defensive_line_variability : CV of last outfield defender x-position over time
      pass_completion_rate       : N_successful / N_passes

    Parameters
    ----------
    events : pd.DataFrame
        Raw StatsBomb events for a single game (from HDF5 events/game_{id}).

    Returns
    -------
    pd.DataFrame  – one row per (possession, possession_team_id)
    """
    events = events[~events["type_name"].isin(EXCLUDE_TYPES)].copy()
    events["time_s"] = events["timestamp"].dt.total_seconds()

    rows = []

    for (poss_id, poss_team_id), grp in events.groupby(
        ["possession", "possession_team_id"], sort=False
    ):
        grp  = grp.sort_values("index")
        att  = grp[grp["team_id"] == poss_team_id]   # possession team
        defe = grp[grp["team_id"] != poss_team_id]   # defending team
        if len(att) == 0:
            continue

        # ── Duration ─────────────────
        t_start = att["time_s"].min()
        T_p     = grp["time_s"].max() - grp["time_s"].min()

        if T_p < 2.0:
            continue

        # ── Ball distances ───────
        D_t, D_f = compute_ball_distances(att)

        # ── Passes ────────────────
        passes   = att[att["type_name"] == "Pass"]
        N_passes = len(passes)
        # A pass is failed when the pass sub-dict contains an "outcome" key
        n_success = sum(
            1 for ex in passes["extra"]
            if not (isinstance(ex, dict) and "outcome" in ex.get("pass", {}))
        )

        # ── Touches ───────────
        T_touches = att["type_name"].isin(TOUCH_TYPES).sum()

        # ── Shot timing ────────
        shots  = att[att["type_name"] == "Shot"]
        t_shot = shots["time_s"].min() if len(shots) > 0 else np.nan

        # ── Entry into attacking third (x > 80) ────────────────
        in_att3 = att[att["location"].apply(
            lambda loc: isinstance(loc, list) and loc[0] > ATT_THIRD_X
        )]
        t_att3 = in_att3["time_s"].min() if len(in_att3) > 0 else np.nan

        att_360 = att[att["freeze_frame_360"].notna()]

        centroids, c_times      = [], []
        att_last_x, def_last_x  = [], []
        spread_vals, sp_times   = [], []
        sp_x_list,  sp_y_list   = [], []
        inter_dist_list         = []

        for _, r in att_360.iterrows():
            att_field, def_field, att_all = parse_freeze_frame(r["freeze_frame_360"])
            t = r["time_s"]

            # Centroid of all possession-team players
            if len(att_all) >= 2:
                centroids.append(np.mean(att_all, axis=0))
                c_times.append(t)

            # Outfield attackers
            if len(att_field) >= 2:
                xs  = [p[0] for p in att_field]
                arr = np.array(att_field)
                att_last_x.append(min(xs))           # most defensive outfield attacker
                sp_x_list.append(np.std(arr[:, 0]))  # longitudinal dispersion
                sp_y_list.append(np.std(arr[:, 1]))  # lateral dispersion
                length = arr[:, 0].max() - arr[:, 0].min()
                width  = arr[:, 1].max() - arr[:, 1].min()
                if width > 0:
                    spread_vals.append(length / width)
                    sp_times.append(t)

                # Inter-team longitudinal distance (requires both sides visible)
                if len(def_field) >= 2:
                    inter_dist_list.append(
                        np.mean([p[0] for p in def_field]) - np.mean([p[0] for p in att_field])
                    )

            # Outfield defenders
            if len(def_field) >= 2:
                # max-x defender = last man / offside line (closest to defending goal)
                def_last_x.append(max(p[0] for p in def_field))

        # Team centroid speed: filter dt > 100 ms to avoid numerical blow-up
        if len(centroids) >= 2:
            c  = np.array(centroids)
            ct = np.array(c_times)
            dt = np.diff(ct)
            d  = np.linalg.norm(np.diff(c, axis=0), axis=1)
            ok = dt > 0.1                              # ≥ 100 ms between frames
            speeds = d[ok] / dt[ok]
            speeds = speeds[speeds < 12.0]
            team_centroid_speed = float(np.mean(speeds)) if len(speeds) > 0 else np.nan
        else:
            team_centroid_speed = np.nan

        N_d = defe["type_name"].isin(DEFENSIVE_TYPES).sum()

        rows.append({
            "game_id":              grp["game_id"].iloc[0],
            "possession":           poss_id,
            "possession_team_id":   poss_team_id,
            "possession_team_name": grp["possession_team_name"].iloc[0],
            "period_id":            grp["period_id"].iloc[0],
            "T_p":                  T_p,
            "N_passes":             N_passes,
            # ── Activity Intensity 
            "passes_per_second":    N_passes / T_p if T_p > 0 else np.nan,
            "distance_per_second":  D_t / T_p     if T_p > 0 else np.nan,
            "distance_per_touch":   D_t / T_touches if T_touches > 0 else np.nan,
            "inter_pass_interval":  _inter_pass_interval(att, passes),
            # ── Directional Intent 
            "attacking_transition_time": (t_att3 - t_start) if (not np.isnan(t_att3) and (t_att3 - t_start) <= 90.0) else np.nan,
            "time_to_shot":              t_shot - t_start if not np.isnan(t_shot) else np.nan,
            "team_centroid_speed":       team_centroid_speed,
            "forward_progression_rate":  D_f / D_t if D_t > 0 else np.nan,
            "attacking_line_variability": _cv(att_last_x),
            "team_spread_acceleration":   _spread_accel(spread_vals, sp_times),
            "inter_team_longitudinal_dist": float(np.mean(inter_dist_list)) if inter_dist_list else np.nan,
            # ── Constraint 
            "pressing_frequency":    N_d / T_p if T_p > 0 else np.nan,
            "spatial_compression_x": float(np.mean(sp_x_list)) if sp_x_list else np.nan,
            "spatial_compression_y": float(np.mean(sp_y_list)) if sp_y_list else np.nan,
            "defensive_line_variability": _cv(def_last_x),
            "pass_completion_rate":  n_success / N_passes if N_passes > 0 else np.nan,
        })

    return pd.DataFrame(rows)

In [7]:
all_features = []

for gid in tqdm.tqdm(games["game_id"].tolist(), desc="Computing pace metrics"):
    try:
        events = pd.read_hdf(spadl_h5, f"events/game_{gid}")
        all_features.append(compute_pace_metrics(events))
    except Exception as e:
        print(f"  Warning: game {gid} skipped – {e}")

features_df = pd.concat(all_features, ignore_index=True)

print(f"Total possessions                    : {len(features_df):,}")
print(f"  with shots                         : {features_df['time_to_shot'].notna().sum():,}")
print(f"  with 360 centroid data             : {features_df['team_centroid_speed'].notna().sum():,}")
print(f"  with inter-team longitudinal dist  : {features_df['inter_team_longitudinal_dist'].notna().sum():,}")
features_df.head()

Computing pace metrics: 100%|██████████| 126/126 [01:31<00:00,  1.37it/s]

Total possessions                    : 21,511
  with shots                         : 2,844
  with 360 centroid data             : 19,177
  with inter-team longitudinal dist  : 21,093


,game_id,possession,possession_team_id,possession_team_name,period_id,T_p,N_passes,passes_per_second,distance_per_second,distance_per_touch,...,team_centroid_speed,forward_progression_rate,attacking_line_variability,team_spread_acceleration,inter_team_longitudinal_dist,pressing_frequency,spatial_compression_x,spatial_compression_y,defensive_line_variability,pass_completion_rate
0,4020846,2,865,England Women's,1,12.583,4,0.317889,9.266345,16.656917,...,4.091170,0.589202,0.215684,0.087919,5.493969,0.317889,10.958952,12.085384,0.067040,1.000000
1,4020846,4,865,England Women's,1,22.175,8,0.360767,10.122333,14.028921,...,4.584197,0.279779,0.294610,0.114591,2.926609,0.135287,9.416492,13.386014,0.240813,0.875000
2,4020846,5,865,England Women's,1,11.770,4,0.339847,15.085112,25.364538,...,4.854883,0.422975,0.209754,0.029774,4.562138,0.169924,8.113141,17.345381,0.082317,0.750000
3,4020846,6,863,Spain Women's,1,45.444,11,0.242056,6.325833,13.066870,...,4.228368,0.470308,0.258318,0.062271,6.003368,0.022005,8.509817,15.546132,0.195632,0.818182
4,4020846,7,865,England Women's,1,55.106,13,0.235909,6.621013,12.161919,...,4.516172,0.502936,0.423941,0.075740,2.617099,0.127028,7.425474,15.020523,0.308447,0.923077


In [8]:
features_df.to_hdf(spadl_h5, key="pace_features", mode="a", complevel=5)
print(f"Saved pace_features → {spadl_h5}  ({len(features_df):,} rows)\n")

metric_groups = {
    "Activity Intensity": [
        "passes_per_second", "distance_per_second",
        "distance_per_touch", "inter_pass_interval",
    ],
    "Directional Intent": [
        "attacking_transition_time", "time_to_shot",
        "team_centroid_speed", "forward_progression_rate",
        "attacking_line_variability", "team_spread_acceleration",
        "inter_team_longitudinal_dist",
    ],
    "Constraint": [
        "pressing_frequency",
        "spatial_compression_x", "spatial_compression_y",
        "defensive_line_variability", "pass_completion_rate",
    ],
}

for group, cols in metric_groups.items():
    print(f"=== {group} ===")
    print(features_df[cols].describe().round(4).to_string())
    print()


Saved pace_features → ../data/women-statsbomb3.h5  (21,511 rows)

=== Activity Intensity ===
       passes_per_second  distance_per_second  distance_per_touch  inter_pass_interval
count         21511.0000           21511.0000          21505.0000           17692.0000
mean              0.3101               7.6278             14.4332               3.5971
std               0.1625               3.7891              9.9822               2.5466
min               0.0000               0.0000              0.0000               0.1270
25%               0.2159               5.5584              9.0527               1.9820
50%               0.2873               7.0748             12.1245               2.9730
75%               0.3728               8.7703             16.1634               4.4670
max               1.4940              51.5744            100.5038              29.5750

=== Directional Intent ===
       attacking_transition_time  time_to_shot  team_centroid_speed  forward_progression_rate  a